In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

import ripser
import persim

In [2]:
########## read the data ##########

processes = ["hhvv", "hhmumu", "jjBG_ptcut", "ttBG_ptcut", "bbBG_ptcut", "wwBG", "wwvvBG", "zzBG", "zzvvBG", "hzvvBG"]
event_type = [str(i)+"_500K" for i in processes]

Xsection = [4.0360050e-02, +3.3945676e-03, 1.345150e+01, 4.423600e+00, 2.325600e+00, 1.506969e+02, 1.224052e+01, 9.618000e+00, 6.473005e+00, 7.4722023e-01]     ### unit: fb
Luminosity = 1000   ### unit: fb^-1
simulation_num = 500000

features = ["VLCjetR10N2", "VLCjetR10N2.PT", "VLCjetR10N2.Eta", "VLCjetR10N2.Phi", "VLCjetR10N2.Mass", "MissingET.MET", "VLCjetR10N2.BTag", "EFlow.Eta", "EFlow.Phi", "EFlow.ET"]

##### set data path #####

event_path = []
for type in event_type:
    event_path.append("/data/mucollider/two_boosted/" + type + "/delphes_output.root")

##### get the data file #####

data_file =[]
for path in event_path:
    data_file.append(uproot.open(path))
    
##### read data with features #####
events = []     ### total events
for process in processes:
    tmp_events = []
    for feature in features:
        tmp_events.append(data_file[processes.index(process)]["Delphes;1"][feature].array())
    tmp_events = np.expand_dims(tmp_events, axis=-1)
    tmp_events = tmp_events.transpose((1,0,2))
    tmp_events = np.squeeze(tmp_events,axis=(2,))
    events.append(tmp_events)
    print("Time:{:^8.4f}(s)".format(time.time()-t1))
del tmp_events

Time:14.7758 (s)
Time:28.1311 (s)
Time:43.8134 (s)
Time:62.0696 (s)
Time:78.4375 (s)
Time:86.9337 (s)
Time:96.1329 (s)
Time:104.8284(s)
Time:114.6925(s)
Time:122.1543(s)


In [3]:
########## define useful function ##########

##### calculate significance #####

def significance(s,b):
    return np.sqrt(2*((s+b)*np.log(1+s/b)-s))

##### count event number #####

def count(events):
    events_num = []
    for i, process in enumerate(processes):
        events_num.append(len(events[processes.index(process)]))
        print("There are", events_num[i], process, "events. Corresponding cross section:", Xsection[processes.index(process)]*events_num[i]/simulation_num, "(fb)")
        
##### select if Fat Jet >= 2 #####
        
def Fat_Jet_selection(events):
    where1 = np.where(events[:,features.index("VLCjetR10N2")]>=2)
    return events[where1]

##### select if M_jet_leading > XX GeV #####

def mass_leading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][0]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][0]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if M_jet_subleading > XX GeV #####

def mass_subleading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][1]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][1]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_leading > XX GeV #####

def pT_leading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][0]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][0]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_subleading > XX GeV #####

def pT_subleading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][1]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][1]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### define X_HH (defined in 2207.09602v2) #####

def calculate_X_HH(event):
    jet_mass1 = event[features.index("VLCjetR10N2.Mass")][0]
    jet_mass2 = event[features.index("VLCjetR10N2.Mass")][1]
    diff1 = np.abs(jet_mass1 - 124)
    diff2 = np.abs(jet_mass2 - 124)
    if diff1<diff2:
        m1 = jet_mass1
        m2 = jet_mass2
    else:
        m1 = jet_mass2
        m2 = jet_mass1
    return np.sqrt(((m1-124)/(0.1*m1+0.00001))**2 + ((m2-115)/(0.1*m2+0.00001))**2)

##### calculate M_JJ #####

def calculate_M_JJ(event):
    p = [0,0,0,0]    ### four momentum
    for i in range(2):    ### leading jet and sub-leading jet
        pt = event[features.index("VLCjetR10N2.PT")][i]
        eta = event[features.index("VLCjetR10N2.Eta")][i]
        phi = event[features.index("VLCjetR10N2.Phi")][i]
        mass = event[features.index("VLCjetR10N2.Mass")][i]
        p[1] = p[1] + pt*np.cos(phi)    ### px
        p[2] = p[2] + pt*np.sin(phi)    ### py
        p[3] = p[3] + pt*np.sinh(eta)    ### pz
        p[0] = p[0] + np.sqrt( mass**2 + (pt*np.cos(phi))**2 + (pt*np.sin(phi))**2 + (pt*np.sinh(eta))**2 )    ### energy
    M_JJ = np.sqrt(p[0]**2 - p[1]**2 - p[2]**2 - p[3]**2)
    return M_JJ

##### calculate pT_JJ #####

def calculate_pT_JJ(event):
    p = [0,0,0,0]    ### four momentum
    for i in range(2):    ### leading jet and sub-leading jet
        pt = event[features.index("VLCjetR10N2.PT")][i]
        eta = event[features.index("VLCjetR10N2.Eta")][i]
        phi = event[features.index("VLCjetR10N2.Phi")][i]
        mass = event[features.index("VLCjetR10N2.Mass")][i]
        p[1] = p[1] + pt*np.cos(phi)    ### px
        p[2] = p[2] + pt*np.sin(phi)    ### py
        p[3] = p[3] + pt*np.sinh(eta)    ### pz
        p[0] = p[0] + np.sqrt( mass**2 + (pt*np.cos(phi))**2 + (pt*np.sin(phi))**2 + (pt*np.sinh(eta))**2 )    ### energy
    pT_JJ = np.sqrt(p[1]**2 + p[2]**2)
    return pT_JJ

##### calculate \Delta_\Eta #####

def calculate_dEta_JJ(event):
    return np.abs(event[features.index("VLCjetR10N2.Eta")][0] - event[features.index("VLCjetR10N2.Eta")][1])

##### calculate TDA #####

def calculate_TDA(event):
#     neutrino_ID = [12, 14, 16, -12, -14, -16]
#     where_jet_particle = np.where(event[features.index("Particle.Status")]==1)[0]
#     where_neutrino = np.where(np.isin(event[features.index("Particle.PID")], neutrino_ID))[0]
#     where_jet_particle = np.delete(where_jet_particle, np.isin(where_jet_particle, where_neutrino))

    particle_ET = event[features.index("EFlow.ET")]
    particle_ETcut = 0
    where_ET_larger = np.where(particle_ET>particle_ETcut)
    particle_ET = event[features.index("EFlow.ET")][where_ET_larger]
    
    center_eta = (event[features.index("VLCjetR10N2.Eta")][0]+event[features.index("VLCjetR10N2.Eta")][1])/2    ### centerize the event
    center_phi = (event[features.index("VLCjetR10N2.Phi")][0]+event[features.index("VLCjetR10N2.Phi")][1])/2
    particle_eta = event[features.index("EFlow.Eta")][where_ET_larger]-center_eta
    particle_phi = event[features.index("EFlow.Phi")][where_ET_larger]-center_phi
    
    where_larger_pi = np.where(particle_phi>np.pi)
    particle_phi[where_larger_pi] = -(2*np.pi - particle_phi[where_larger_pi])    ### check if jet particle split
    where_smaller_mpi = np.where(particle_phi<-np.pi)
    particle_phi[where_smaller_mpi] = 2*np.pi + particle_phi[where_smaller_mpi]    ### check if jet particle split

    P = np.array([particle_eta, particle_phi]).T   ### calculate TDA
    diagrams = ripser.ripser(P)['dgms']
    
    return diagrams

##### calculate TDA sum of lifetime #####

def TDA_sum(diagrams):
    sum_l0 = np.sum(diagrams[0][:-1,1] - diagrams[0][:-1,0])
    sum_l1 = np.sum(diagrams[1][:,1] - diagrams[1][:,0])
    return sum_l0, sum_l1

##### calculate TDA mean of lifetime #####

def TDA_mean(diagrams):
    l0 = diagrams[0][:-1,1]-diagrams[0][:-1,0]
    l1 = diagrams[1][:,1]-diagrams[1][:,0]
    if np.sum(l1)==0:
        l1 = np.concatenate([l1,[0]])
    return np.mean(l0), np.mean(l1)

##### calculate TDA entropy #####

def TDA_entropy(diagrams):
    l0 = diagrams[0][:-1,1]-diagrams[0][:-1,0]
    L0 = np.sum(l0)
    S0 = (l0/L0)*np.log2((l0/L0))
    l1 = diagrams[1][:,1]-diagrams[1][:,0]
    L1 = np.sum(l1)
    S1 = (l1/L1)*np.log2((l1/L1))
    return -np.sum(S0), -np.sum(S1)

##### calculate TDA Lifetime*Birthtime #####

def TDA_LB(diagrams):
    return np.sum(diagrams[1][:,0]*(diagrams[1][:,1]-diagrams[1][:,0]))

In [4]:
########## preselection ##########

print("Before any selection:")
count(events)

##### 2 fat jet selection #####

for process in processes:
    events[processes.index(process)] = Fat_Jet_selection(events[processes.index(process)])
print("\nAfter 2 fat jet selection:")
count(events)

##### leading jet pT selection #####

leading_low_pT_cut = 200   ### GeV
leading_high_pT_cut = 800   ### GeV
for process in processes:
    events[processes.index(process)] = pT_leading_selection(events[processes.index(process)], leading_low_pT_cut, leading_high_pT_cut)
print("\nAfter leading jet pT selection:")
count(events)

##### subleading jet pT selection #####

subleading_low_pT_cut = 200   ### GeV
subleading_high_pT_cut = 600   ### GeV
for process in processes:
    events[processes.index(process)] = pT_subleading_selection(events[processes.index(process)], subleading_low_pT_cut, subleading_high_pT_cut)
print("\nAfter subleading jet pT selection:")
count(events)

##### leading jet mass selection #####

leading_low_mass_cut = 65
leading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_leading_selection(events[processes.index(process)], leading_low_mass_cut, leading_high_mass_cut)
print("\nAfter leading jet mass selection:")
count(events)

##### subleading jet mass selection #####

subleading_low_mass_cut = 65
subleading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_subleading_selection(events[processes.index(process)], subleading_low_mass_cut, subleading_high_mass_cut)
print("\nAfter subleading jet mass selection:")
count(events)

Before any selection:
There are 500000 hhvv events. Corresponding cross section: 0.04036005 (fb)
There are 500000 hhmumu events. Corresponding cross section: 0.0033945676 (fb)
There are 500000 jjBG_ptcut events. Corresponding cross section: 13.4515 (fb)
There are 500000 ttBG_ptcut events. Corresponding cross section: 4.4236 (fb)
There are 500000 bbBG_ptcut events. Corresponding cross section: 2.3256 (fb)
There are 500000 wwBG events. Corresponding cross section: 150.6969 (fb)
There are 500000 wwvvBG events. Corresponding cross section: 12.24052 (fb)
There are 500000 zzBG events. Corresponding cross section: 9.618 (fb)
There are 500000 zzvvBG events. Corresponding cross section: 6.473005 (fb)
There are 500000 hzvvBG events. Corresponding cross section: 0.7472202300000002 (fb)

After 2 fat jet selection:
There are 500000 hhvv events. Corresponding cross section: 0.04036005 (fb)
There are 500000 hhmumu events. Corresponding cross section: 0.0033945676 (fb)
There are 500000 jjBG_ptcut even

In [5]:
##### calculate the high level features #####

num_of_events = []

m_J_leading = []
m_J_subleading = []
pt_J_leading = []
pt_J_subleading = []
missET = []
X_HH = []
M_JJ = []
pT_JJ = []
dEta_JJ = []
BTag_leading = []
BTag_subleading = []

for process in processes:
    num_of_events.append(len(events[processes.index(process)]))
    for i in range(len(events[processes.index(process)])):
        m_J_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.Mass")][0])
        m_J_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.Mass")][1])
        pt_J_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.PT")][0])
        pt_J_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.PT")][1])
        missET.append(events[processes.index(process)][i][features.index("MissingET.MET")][0])
        X_HH.append(calculate_X_HH(events[processes.index(process)][i]))
        M_JJ.append(calculate_M_JJ(events[processes.index(process)][i]))
        pT_JJ.append(calculate_pT_JJ(events[processes.index(process)][i]))
        dEta_JJ.append(calculate_dEta_JJ(events[processes.index(process)][i]))
        BTag_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.BTag")][0])
        BTag_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.BTag")][1])
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:212.3597(s)
Time:228.3507(s)
Time:232.3889(s)
Time:236.4362(s)
Time:240.8242(s)
Time:246.5277(s)
Time:257.6752(s)
Time:264.5431(s)
Time:278.0322(s)
Time:292.9937(s)


In [6]:
##### calculate TDA high level features #####

H_0 = []    ### sum of 0'th order lifetime 
H_1 = []    ### sum of 1'th order lifetime 
S_0 = []    ### sum of 0'th order entropy
S_1 = []    ### sum of 1'th order entropy
LB_1 = []   ### sum of 1'th order Lifetime*Birthtime

for process in processes:
    for i in range(len(events[processes.index(process)])):
        if (i%20000)==19999:
            print("check point")
        TDA_tmp = calculate_TDA(events[processes.index(process)][i])
        H_0.append(TDA_sum(TDA_tmp)[0])        
        H_1.append(TDA_sum(TDA_tmp)[1])
        S_0.append(TDA_entropy(TDA_tmp)[0])   
        S_1.append(TDA_entropy(TDA_tmp)[1])
        LB_1.append(TDA_LB(TDA_tmp))
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

check point
check point
check point
check point
check point
check point
check point
Time:752.6110(s)
check point
check point
check point
check point
check point
check point
check point
Time:1194.1825(s)
check point
Time:1421.4019(s)
check point
Time:1698.8401(s)
check point
Time:1959.3859(s)
check point
check point
Time:2042.6309(s)
check point
check point
check point
check point
Time:2217.5827(s)
check point
check point
check point
Time:2332.9777(s)
check point
check point
check point
check point
check point
check point
Time:2572.4338(s)
check point
check point
check point
check point
check point
check point
Time:2906.6741(s)


In [7]:
##### make target #####

labels = np.arange(len(num_of_events))
target_everytype = np.repeat(labels, num_of_events)    ### this target 0~8 represent ["hhvv", "hhmumu", "jjBG_ptcut", "ttBG_ptcut", "bbBG_ptcut", "wwBG", "wwvvBG", "zzBG", "zzvvBG"]

target_sigbg = np.zeros(sum(num_of_events))
sig_num = num_of_events[0]+num_of_events[1]
target_sigbg[0:sig_num] = 1     ### this target 1 represent signal, 0 represent background

In [8]:
##### output test data #####

d = {'m_J_leading':m_J_leading, 'm_J_subleading':m_J_subleading, 'pt_J_leading':pt_J_leading, 'pt_J_subleading':pt_J_subleading, 'missET':missET, 'X_HH':X_HH, 'M_JJ':M_JJ, 'pT_JJ':pT_JJ, 'dEta_JJ':dEta_JJ, 'BTag_leading':BTag_leading, 'BTag_subleading':BTag_subleading, 'H_0':H_0, 'H_1':H_1, 'S_0':S_0, 'S_1':S_1, 'LB_1':LB_1, 'target_sigbg':target_sigbg, 'target_everytype':target_everytype}
df = pd.DataFrame(data=d)
df

,m_J_leading,m_J_subleading,pt_J_leading,pt_J_subleading,missET,X_HH,M_JJ,pT_JJ,dEta_JJ,BTag_leading,BTag_subleading,H_0,H_1,S_0,S_1,LB_1,target_sigbg,target_everytype
0,126.326309,108.387589,386.186310,245.464920,278.663544,0.637258,903.569668,282.219045,1.872866,6,7,8.216068,0.047200,5.279780,2.426942,0.003951,1.0,0
1,118.044922,111.297234,389.527374,350.259674,106.212425,0.604300,960.274007,112.899464,1.434012,4,7,11.382067,0.103302,5.407406,2.743507,0.020597,1.0,0
2,113.501045,99.448242,352.300873,334.518677,214.195816,1.816898,920.117233,214.737470,1.597468,7,6,7.810970,0.111604,4.567818,2.780592,0.018763,1.0,0
3,126.547104,124.887665,347.181732,322.873688,106.539497,0.915238,848.598657,108.905742,1.305463,5,5,8.243116,0.056814,4.686552,1.346818,0.022817,1.0,0
4,127.247276,96.656952,286.092255,270.694122,83.730888,1.914827,600.846529,79.038071,0.281423,5,0,8.436831,0.199417,5.684320,2.606929,0.039441,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866054,124.713478,77.110153,328.826965,302.863647,96.518463,4.914057,1431.341327,96.843732,2.889620,7,0,13.309820,0.579264,5.354273,2.746487,0.235679,0.0,9
866055,74.198212,87.886055,391.648285,302.423035,354.598907,6.864729,758.474519,357.683242,1.201538,4,0,8.476855,0.105559,3.897012,0.859171,0.030724,0.0,9
866056,105.105865,77.649590,321.398163,265.746338,159.964294,5.135045,939.289637,159.384784,2.088760,6,0,8.761282,0.032773,4.615244,0.189123,0.006347,0.0,9
866057,89.882179,129.630142,518.687256,510.653748,144.216888,2.828074,1094.635928,140.909336,0.611172,6,5,7.740872,0.054341,4.052139,2.932016,0.008379,0.0,9


In [9]:
h5f = h5py.File('/data/mucollider/two_boosted/data_HLfeature.h5', 'w')
for key, value in d.items():
    h5f.create_dataset(key, data=value)
    print(f"已寫入 Dataset: {key}")

已寫入 Dataset: m_J_leading
已寫入 Dataset: m_J_subleading
已寫入 Dataset: pt_J_leading
已寫入 Dataset: pt_J_subleading
已寫入 Dataset: missET
已寫入 Dataset: X_HH
已寫入 Dataset: M_JJ
已寫入 Dataset: pT_JJ
已寫入 Dataset: dEta_JJ
已寫入 Dataset: BTag_leading
已寫入 Dataset: BTag_subleading
已寫入 Dataset: H_0
已寫入 Dataset: H_1
已寫入 Dataset: S_0
已寫入 Dataset: S_1
已寫入 Dataset: LB_1
已寫入 Dataset: target_sigbg
已寫入 Dataset: target_everytype


In [10]:
print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:2913.4562(s)


In [11]:
h5f.close()